# gras — two races, with the engine talking

`gras` searches neural architectures while they train: a population of candidate
nets shares one seeded stream, is scored every step, and the weak ones are culled
as soon as they fall behind.

This notebook runs the binding's **two arms** at their smallest useful size — a
supervised race on MNIST and a reinforcement-learning race on CartPole — and lets
the engine print everything it normally prints. That log *is* the point:
`log_level="summ"` prints the same cards the Rust examples print, because it is
the same code. Nothing below is re-rendered for Python.

**Every code cell is self-contained.** Imports, seed, sizes and paths are declared
inside the cell that uses them, so you can run any cell on its own, in any order,
without running the ones above it. There is no setup cell that everything secretly
depends on.

**Before running**

- `gras._core` must be built for *this* interpreter: `maturin develop`, see the
  README. The extension links libtorch; `import gras` finds it on its own.
- `data/mnist/train` must exist. `data/` is gitignored, so a fresh clone has the
  code but no datasets.
- CartPole needs `pip install gymnasium`.
- If this kernel has already run a race with `log_level="none"`, **restart the
  kernel**: the log ceiling is process-wide and set once, by the first engine
  (notes/PYTHON_BINDINGS.md §4.6). Clearing the namespace does not undo it.

## 0. Is this machine ready?

A pure check: it prints what it found and hands nothing to the cells below, so it
is safe to run first, last, or not at all.

In [5]:
# Nothing in this cell is reused below — that is why it can print and stop.
from pathlib import Path

import gras

# Where is the checkout? Walk up from the kernel's cwd until Cargo.toml appears.
here = Path.cwd().resolve()
root = next((p for p in (here, *here.parents) if (p / "Cargo.toml").is_file()), None)
mnist = root / "data" / "mnist" / "train" if root else None

# The backend is baked into the compiled extension at BUILD time; it cannot be
# switched at runtime.
print(f"gras      {gras.__version__}  from {Path(gras.__file__).parent}")
print(f"backend   {'cuda' if gras.built_with_cuda else 'cpu'}  (fixed when the extension was built)")
print(f"checkout  {root or 'NOT FOUND — start the kernel inside the repo, or cd there'}")
if mnist is None:
    print("mnist     unknown (no checkout)")
else:
    print(f"mnist     {mnist.relative_to(root)} present: {mnist.is_dir()}"
          + ("" if mnist.is_dir() else "  — data/ is gitignored; §1 needs it"))

gras      0.1.0  from /home/maulb/gras/python/gras
backend   cpu  (fixed when the extension was built)
checkout  /home/maulb/gras
mnist     data/mnist/train present: True


## 1. MNIST — the supervised arm

784 inputs → 10 classes, 60k rows, one-hot targets. The engine reads the directory
and infers both dimensions; there is no separate shape declaration. That directory
is a **dataset spec**: `inputs.bin` + `targets.bin` (what `save_dataset` and
`RaceEngine.from_arrays` write), or `inputs.csv` + `targets.csv`, converted once
and cached inside it.

One cell does the whole thing: declare the knobs → race → read back what the race
wrote. Run it alone and it works.

**What to read in the log**, top to bottom: `Race init` → `Run manifest` (the recipe
the run records, and therefore what a resume must match) → `Legend` (what every
column name means) → `Search space` (what the GP may invent) → `Trainer` → one
`step N │ …` line per clock → `Elite index` / `Elite topology` / `Elite weights` →
`Stop` → `Artifacts & resume` → `Run accounting` → `Final field`, one row per net.

In [6]:
# ── 1. Everything this cell needs ────────────────────────────────────────────
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import gras
import pandas as pd
from gras import RaceConfig, RaceEngine

# The build is fixed at compile time, so the honest thing is to check the claim
# against the artifact you actually imported, and stop if they disagree.
CUDA = False          # rebuild with `--features cuda` for True (see README)
assert gras.built_with_cuda == CUDA, (
    f"this extension was built with cuda={gras.built_with_cuda}; rebuild it to match "
    f"CUDA={CUDA} (see README, Python section)"
)

SEED = 42            # same seed + same config ⇒ same history.csv, byte for byte
MNIST_POP = 16       # candidate nets competing at once
MNIST_STEPS = 6      # training clocks. The first 5 are newborn shields, so shorter
                     # races never cull or breed and the log is all prologue.

# Walk up from the kernel's cwd until data/mnist/train shows up. `data/` is
# gitignored, so a fresh clone has the code but no datasets — fail by name.
MNIST_DIR = next(
    (p / "data" / "mnist" / "train" for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "mnist" / "train").is_dir()),
    None,
)
if MNIST_DIR is None:
    raise RuntimeError("data/mnist/train not found above the kernel's cwd "
                       "(data/ is gitignored — see the README)")

# ── 2. Race ──────────────────────────────────────────────────────────────────
# A throwaway dir per run, so a re-run never resumes the previous one by accident.
mnist_run = Path(tempfile.mkdtemp(prefix="gras-mnist-"))

cfg = RaceConfig(
    pop_size=MNIST_POP,      # candidate nets competing at once
    elite_count=2,           # the best N are protected and saved at stop
    max_steps=MNIST_STEPS,   # training clocks the race runs
    log_level="summ",        # the full card set; "none" silences it completely
)

# NOTE: the engine takes its own copy of the config when it is built, so
# `log_level` has to be set HERE. Setting `cfg.log_level` after this line changes
# nothing, and the run stays quiet while you wonder why.
engine = RaceEngine(
    MNIST_DIR,               # dataset directory (dims are inferred from it)
    cfg,
    seed=SEED,               # the whole race repeats from this number
    run_dir=mnist_run,       # without it: results/<timestamp> in the kernel's cwd
)

print(f"run dir {engine.run_dir}  seed {engine.run_seed}")
stop = engine.run()          # the log above is this call
print(f"stop    {stop} after {engine.step_clock} clocks")

# ── 3. Read back what it wrote (same dir the engine just returned) ───────────
run_dir = Path(engine.run_dir)
history = pd.read_csv(run_dir / "history.csv")
header = json.loads((run_dir / "engine.json").read_text())

print(f"\n{len(history)} readings across {history['hash'].nunique()} nets")
print(f"recipe  mode={header['engine_mode']} fitness={header['fitness_label']} "
      f"{header['input_dim']}→{header['output_dim']} seed={header['run_seed']}")

# One row per net per clock: train_loss is what it learned on, eval_loss on rows it
# never trained on, fitness is the ranking value (here: eval cross-entropy — lower
# is better, which is what `fitness_direction` in engine.json says).
print(history.groupby("step")[["train_loss", "eval_loss", "fitness"]].mean().round(3))

best = history.loc[history["fitness"].idxmin()]
print(f"best net {best['hash']} at clock {int(best['step'])}: fitness {best['fitness']:.3f}")

# Each net's own file: who it is, where it came from, and the blueprint the race
# evolved — no weights in this JSON (they live in the .safetensors beside it).
envelope = json.loads(sorted((run_dir / "nets").glob("*.json"))[0].read_text())
print(f"net {envelope['hash']}: {envelope['meta']['params']} parameters, "
      f"from {envelope['created_from']}, alive={envelope['is_alive']}")

┌─────────────┐
│ ▸ Race init │
└─────────────┘
┌──────────────┬──────────────────────────┐
│ live nets    ┆ 16                       │
│ run dir      ┆ /tmp/gras-mnist-niscap9w │
│ seeded slots ┆ 0                        │
└──────────────┴──────────────────────────┘
╰─ all-random founder batch
run dir /tmp/gras-mnist-niscap9w  seed 42
┌────────────────┐
│ ▸ Run manifest │
└────────────────┘
┌────────────────────────────┬───────────────────────────────────────────────┐
│ run_id                     ┆ 1793306038114                                 │
│ seed                       ┆ 42                                            │
│ pid                        ┆ 196504                                        │
│ mode                       ┆ tabular                                       │
│ pop_size                   ┆ 16                                            │
│ max_steps                  ┆ 6                                             │
│ smoothing_window (K)       ┆ 10                   

## 2. CartPole — the reinforcement-learning arm

Same `RaceConfig`, same seed promise; only the **source of experience** changes.
An environment replaces the labelled dataset: each match is one episode, the reward
is survival, and the trainer is the crate's `Reinforce`. Nothing about CartPole is
compiled into `gras` — it arrives through the `Env` adapter, which is why any
gym/gymnasium environment with a discrete action space plugs into this cell.

Again one cell does it all, and it shares nothing with §1 — not the imports, not
the seed, not a path. Run it in a fresh kernel if you like.

**Why 20 × 10.** RL spends a real environment per candidate: ~1.1 s per clock at
pop 20, so ten clocks is ~13 s. Ten is also the first size where the interesting
log appears — culls, crossover and mutation rolls, seat terms, newborn shields —
because the opening 5-clock shields expire at clock 5.

`risk_aversion=0.5` is set on purpose: it ranks a net by `avg − λ·spread`, so a net
steady at 25 turns beats one that swings between 10 and 40. `score↑` in the step
line shows that arithmetic with the charge broken out.

In [ ]:
# ── 1. Everything this cell needs (deliberately repeated from §1) ────────────
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import gras
import pandas as pd
from gras import RaceConfig, RaceEngine

try:
    import gymnasium
except ImportError as e:                      # the only extra dependency here
    raise RuntimeError("pip install gymnasium — the RL arm needs an environment") from e

CUDA = False          # same build check as §1: the backend is fixed at build time
assert gras.built_with_cuda == CUDA, (
    f"this extension was built with cuda={gras.built_with_cuda}; rebuild it to match "
    f"CUDA={CUDA} (see README, Python section)"
)

SEED = 42            # the same seed, but a different arm: RL needs its own run
CART_POP = 20        # ~1.1 s per clock at this size, so keep the count low
CART_STEPS = 10      # clocks; shields expire at 5, so 10 shows real evolution

# ── 2. Race ──────────────────────────────────────────────────────────────────
cart_run = Path(tempfile.mkdtemp(prefix="gras-cartpole-"))

cart_cfg = RaceConfig(
    pop_size=CART_POP,
    elite_count=2,
    max_steps=CART_STEPS,
    log_level="summ",        # set HERE: the engine copies the config at build time
    risk_aversion=0.5,       # rank by avg − 0.5 × spread: prefer steady nets
)

# `gym(...)` is the only structural difference from §1: an env instead of a
# directory. The observation width and the action count are probed from the env.
cart = RaceEngine.gym(
    gymnasium.make("CartPole-v1"),
    cart_cfg,
    seed=SEED,
    run_dir=cart_run,
)

print(f"run dir {cart.run_dir}  seed {cart.run_seed}")
stop = cart.run()            # the log above is this call
print(f"stop    {stop} after {cart.step_clock} clocks")

# ── 3. Read back: same file layout, different meaning of the numbers ─────────
run_dir = Path(cart.run_dir)
history = pd.read_csv(run_dir / "history.csv")
header = json.loads((run_dir / "engine.json").read_text())

print(f"\n{len(history)} readings across {history['hash'].nunique()} nets")
print(f"recipe  mode={header['engine_mode']} fitness={header['fitness_label']} "
      f"direction={header['fitness_direction']} seed={header['run_seed']}")

# Higher is better here: these are episode lengths in turns.
print(history.groupby("step")["fitness"].agg(["min", "mean", "max"]).round(1))
print(f"longest episode in the race: {history['fitness'].max():.0f} turns "
      f"(a coin-flip policy averages ~20)")

┌─────────────┐
│ ▸ Race init │
└─────────────┘
┌──────────────┬─────────────────────────────┐
│ live nets    ┆ 20                          │
│ run dir      ┆ /tmp/gras-cartpole-55mxewi6 │
│ seeded slots ┆ 0                           │
└──────────────┴─────────────────────────────┘
╰─ all-random founder batch
run dir /tmp/gras-cartpole-55mxewi6  seed 42
┌────────────────┐
│ ▸ Run manifest │
└────────────────┘
┌────────────────────────────┬───────────────────────────────────────────────┐
│ run_id                     ┆ 1793306038937                                 │
│ seed                       ┆ 42                                            │
│ pid                        ┆ 196504                                        │
│ mode                       ┆ rl                                            │
│ pop_size                   ┆ 20                                            │
│ max_steps                  ┆ 10                                            │
│ smoothing_window (K)       ┆ 10 

## 3. In-memory arrays — and a fitness that is not the loss

`from_arrays` takes NumPy (or anything with the buffer protocol) straight to the
crate's native `inputs.bin` + `targets.bin`, then builds the engine exactly as the
directory form does — same inference, same artifacts.

It is also the cheapest place to see that **training and ranking are two
decisions**: `loss="mse"` is what the optimizer minimises, `fitness="mae"` is
what the race ranks on, and `metrics=["r2"]` is a third number that is recorded
and never selected on. The `Run manifest` above prints all three.

No MNIST needed: 512 rows of 8 features, and `Y` is one column wide because MSE
takes the shape you hand it (cross-entropy is the loss that insists on one-hot
rows).

In [ ]:
# ── 1. Everything this cell needs ─────────────────────────────────────────────
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import gras
from gras import RaceConfig, RaceEngine

# The backend is baked into the extension at build time (see §1).
CUDA = False
assert gras.built_with_cuda == CUDA, (
    f"this extension was built with cuda={gras.built_with_cuda}; rebuild it to match "
    f"CUDA={CUDA} (see README, Python section)"
)

SEED = 7
ARRAYS_POP = 8
ARRAYS_STEPS = 6

# ── 2. Data: a plain regression problem, shaped for a 1-column target ────────
rng = np.random.default_rng(0)
X = rng.random((512, 8), dtype=np.float32)                 # 512 rows × 8 features
w = rng.normal(size=(8, 1)).astype(np.float32)
Y = (X @ w).astype(np.float32)                             # (512, 1) — NOT flat

work = Path(tempfile.mkdtemp(prefix="gras-arrays-"))
data_dir = work / "data"        # from_arrays creates it and writes the native pair

# ── 3. Race: MSE trains, MAE ranks, R² is only recorded ──────────────────────
cfg = RaceConfig(
    pop_size=ARRAYS_POP,
    elite_count=2,
    max_steps=ARRAYS_STEPS,
    log_level="summ",           # set HERE: the engine copies the config at build time
)

race = RaceEngine.from_arrays(
    X, Y, data_dir, cfg,
    seed=SEED,
    loss="mse",                 # what the optimizer minimises
    fitness="mae",              # what the race RANKS on — a different question
    metrics=["r2"],             # recorded in history.csv; selection never sees it
    lr=2e-3,
)

print(f"run dir {race.run_dir}  seed {race.run_seed}")
stop = race.run()               # the log above is this call
print(f"stop    {stop} after {race.step_clock} clocks")

# ── 4. Read back what it wrote ───────────────────────────────────────────────
run_dir = Path(race.run_dir)
history = pd.read_csv(run_dir / "history.csv")
header = json.loads((run_dir / "engine.json").read_text())

print(f"\nfrom_arrays wrote {sorted(p.name for p in data_dir.iterdir())}")
print(f"recipe  loss={header['trainer']['loss']}  fitness={header['fitness_label']}  "
      f"metrics={header['informative_metrics']}")
print(history.groupby("step")[["train_loss", "fitness", "r2"]].mean().round(3))
print("train_loss is the MSE it minimises; fitness is the MAE it is ranked by;")
print("r2 is the informative third number — recorded, never ranked on.")

## 4. A race that builds on the last

`engine.guardrail()` is the bridge between two runs. It scores every elite on
**fresh draws from the run's eval pool** — the 20% split no train batch ever
touches — through the run's own fitness, and records the champion in
`engine.json`.

That record is what `elite_previous_load` looks for. With
`elite_previous_dir=<source run>`, the next race seeds its first founder slots
from the source's ranked elites, best first, capped by `pop_size`: watch for
`seeded slots 2` in `Race init` and `ranks inherited 1..2` in the `Inheritance`
card. TOPOLOGY only — weights restart from the stored topology seed, so a
champion blueprint moves, its training does not.

A source with **no** guardrail record seeds nothing, and the card says so. That
is the whole gate: the knob was always wired; `guardrail()` is what gives it
something to pass.

In [ ]:
# ── 1. Everything this cell needs ─────────────────────────────────────────────
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import gras
from gras import RaceConfig, RaceEngine

CUDA = False          # same build check as the other cells
assert gras.built_with_cuda == CUDA, (
    f"this extension was built with cuda={gras.built_with_cuda}; rebuild it to match "
    f"CUDA={CUDA} (see README, Python section)"
)

SEED = 5
CHAIN_POP = 6        # small on purpose: two races in one cell
CHAIN_STEPS = 4      # enough for a ranking, short of a cull
EXAMS = 8            # holdout draws per elite (guardrail_matches)

# Walk up from the kernel's cwd for the tiny blobs dataset (2 → 3).
BLOBS = next(
    (p / "data" / "blobs" for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "blobs").is_dir()),
    None,
)
if BLOBS is None:
    raise RuntimeError("data/blobs not found above the kernel's cwd "
                       "(data/ is gitignored — see the README)")

work = Path(tempfile.mkdtemp(prefix="gras-chain-"))
source_dir, child_dir = work / "source", work / "child"

# ── 2. The source race, and the exam that certifies it ───────────────────────
source = RaceEngine(
    BLOBS,
    RaceConfig(pop_size=CHAIN_POP, elite_count=2, max_steps=CHAIN_STEPS,
               guardrail_matches=EXAMS,   # how many unseen draws per elite
               log_level="summ"),
    seed=SEED,
    run_dir=source_dir,
)
source.run()
verdict = source.guardrail()      # the Guardrail card above, and the record below

print(f"certified {verdict['hash'][:8]} on {verdict['games']} unseen draws: "
      f"holdout {verdict['holdout_mean']:.3f} ± {verdict['holdout_se']:.3f} "
      f"vs race {verdict['race_smoothed']:.3f}")

record = json.loads((source_dir / "engine.json").read_text())["guardrail"]
print(f"engine.json guardrail: champion {record['champion_hash'][:8]} · "
      f"{record['matches_scored']} draws · holdout mean {record['holdout_mean']:.3f}")

# ── 3. The next race: new dir, founders seeded from the source ───────────────
child = RaceEngine(
    BLOBS,
    RaceConfig(
        pop_size=CHAIN_POP,
        elite_count=2,
        max_steps=CHAIN_STEPS,
        guardrail_matches=0,       # no exam needed: this cell only reads what it inherited
        elite_previous_load=True,  # OFF by default in Python (unlike the crate)
        elite_previous_dir=source_dir,
        log_level="summ",
    ),
    seed=SEED + 1,                 # a different seed: this is a new experiment
    run_dir=child_dir,
)
child.run()

# ── 4. Read back the lineage ─────────────────────────────────────────────────
src = json.loads((child_dir / "engine.json").read_text())["run_topology_source"]
print(f"\nchild inherited {len(src['elite_hashes'])} blueprint(s) from run "
      f"{src['source_run_id']}: champion {src['champion_hash'][:8]}, "
      f"holdout {src['guardrail_mean']:.3f}")

for path in sorted((child_dir / "nets").glob("*.json")):
    doc = json.loads(path.read_text())
    if str(doc["created_from"]).startswith("seed:"):
        print(f"  {doc['hash'][:8]} created_from {doc['created_from']} "
              f"({doc['meta']['params']} parameters)")

## 5. Determinism, across processes

The Rust contract is the product claim: same `run_seed` + same config ⇒ a
bit-identical run. `history.csv` is the stream that proves it, so the test hashes
that file.

Each race here runs in its **own interpreter** with `log_level="none"` (two
identical race logs would only bury the numbers). That also makes the claim
stronger than a same-process loop: three separate processes, two of them seeded
`42`, must emit the same digest — and a third seeded `43` must not.

In [ ]:
# ── 1. Everything this cell needs ─────────────────────────────────────────────
from __future__ import annotations

import subprocess
import sys
import tempfile
from pathlib import Path

SEED = 42
DET_POP = 3          # a race only has to be long enough to write a stream
DET_STEPS = 2

BLOBS = next(
    (p / "data" / "blobs" for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "blobs").is_dir()),
    None,
)
if BLOBS is None:
    raise RuntimeError("data/blobs not found above the kernel's cwd "
                       "(data/ is gitignored — see the README)")

# The race, in a child interpreter: quiet there, so the log ceiling of THIS
# kernel is never touched (that ceiling is set once, by the first engine).
CHILD = [
    "import hashlib, sys",
    "from pathlib import Path",
    "from gras import RaceConfig, RaceEngine",
    "data, seed, run_dir = sys.argv[1], int(sys.argv[2]), Path(sys.argv[3])",
    "cfg = RaceConfig(pop_size=3, elite_count=1, max_steps=2, log_level='none')",
    "race = RaceEngine(data, cfg, seed=seed, run_dir=run_dir)",
    "race.run()",
    "print(hashlib.sha256((run_dir / 'history.csv').read_bytes()).hexdigest())",
]


def digest(seed: int) -> str:
    """One whole race in a FRESH process; return the history.csv fingerprint."""
    run_dir = Path(tempfile.mkdtemp(prefix=f"gras-detect-{seed}-"))
    done = subprocess.run(
        [sys.executable, "-c", "\n".join(CHILD), str(BLOBS), str(seed), str(run_dir)],
        capture_output=True,
        text=True,
        check=True,
    )
    return done.stdout.strip().splitlines()[-1][:16]


first, again, other = digest(SEED), digest(SEED), digest(SEED + 1)

print(f"seed {SEED:>3}  {first}")
print(f"seed {SEED:>3}  {again}   same seed  → same digest: {first == again}")
print(f"seed {SEED + 1:>3}  {other}   other seed → different digest: {other != first}")

## Reading the log without a tutorial

- `ran 18 of 20 = 17 ticketed + 1 crown` — who trained this clock. The crown is the
  net acting without updating; `held back` is newborn shields, plus lottery idles
  when `train_merit (Ω)` is above 0.
- `score↑ = avg − λ·spread` — the single ranking number. `avg` is what the net
  actually scored; the charge is what it is docked for being unreliable.
- `seats 2 of 2`, `seat term complete` — the elite seats rotate on a term, so one
  lucky net cannot hold the crown forever.
- `crossover roll 3 of 10 → child … │ culled …` — evolution fired, and which net it
  replaced.
- `Final field` — every net's row: fitness, tickets, updates, idle clocks,
  inefficiency, and the gates (`b` born shield, `j` join window, `t` crown term).

- `Guardrail` — the exam that certifies a champion on unseen draws; it prints
  whatever `log_level` says, because it is an explicit call, not a race card. Its
  record in `engine.json` is what the next run inherits from.
- `Inheritance` — `seeded slots N` / `ranks inherited …`, or the sentence naming
  why nothing was inherited (a source with no certified champion seeds nothing).

Where the rest lives: `README.md` (the surface), `notes/PYTHON_BINDINGS.md`
(determinism, the guardrail, and packaging), `TODO.md` (what is deliberately
absent — per-step control, Python callbacks, and `resume()`).

The run directories are throwaway temp dirs, one per cell run. Pass
`run_dir=Path("results/mine")` to keep one, but note that a second run pointed at
the same directory does **not** resume it: resume is Rust-only today
(`TabularEngine::resume`), so a repeated `run_dir` starts a fresh race and
overwrites the artifacts. Continuing a run from Python means inheritance (§4): a
new run in a new directory whose founders come from a certified one. Because each
cell declares its own `run_dir`, a cell never picks up another cell's run.